# Ingestion du RPG dans PostGIS

*SeineCrops — notebook d'acquisition de la donnée socle*

Ce notebook charge le **Registre parcellaire graphique (RPG)** — la vérité terrain du projet — 
dans une base **PostGIS**, de façon **rejouable** et **traçable**, dans l'esprit opérationnel 
inspiré du 3STR.

| # | Section | État |
|---|---|---|
| 1 | Récupération du millésime | ✅ |
| 2 | Reconnaissance | ✅ |
| 3 | Préparation de PostGIS | ✅ |
| 4 | Découpe spatiale sur l'AOI | ✅ |
| 5 | Chargement | à venir |
| 6 | Reprojection (UTM 31N) | à venir |
| 7 | Validation / QA | à venir |
| 8 | Documentation / métadonnées | à venir |

## Imports et paramètres globaux

Cellule unique, à exécuter en premier : tous les imports et paramètres utilisés 
dans les sections 1 à 4 sont regroupés ici, pour que le notebook soit rejouable 
section par section sans `NameError`.

In [ ]:
from __future__ import annotations

import datetime
import hashlib
import json
import os
import re
import subprocess
import urllib.request
import xml.etree.ElementTree as ET
from datetime import date
from pathlib import Path

import fiona
import geopandas as gpd
import pandas as pd
import psycopg2
import pyogrio
from dotenv import load_dotenv


# ── Racine du projet (indépendante du dossier d'exécution du notebook) ────
def trouver_racine(marqueurs: tuple[str, ...] = (".projectroot", ".git", "pyproject.toml")) -> Path:
    """Remonte depuis le cwd jusqu'au premier dossier contenant l'un des marqueurs.

    Évite que les chemins relatifs se résolvent par rapport à notebooks/.
    Marqueur recommandé : un fichier vide `.projectroot` à la racine du dépôt.
    """
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if any((parent / m).exists() for m in marqueurs):
            return parent
    raise FileNotFoundError(
        f"Racine introuvable (aucun marqueur {marqueurs} en remontant depuis {base})."
    )


ROOT     = trouver_racine()
DATA_DIR = ROOT / "data"

# ── Choix primaire : la campagne / le millésime de référence ─────────────
# C'est la PÉRIODE d'observation qui est choisie en premier (cf. cadrage) ;
# le millésime RPG et la sélection Sentinel-2 en DÉCOULENT.
ANNEE_REFERENCE = 2024                 # campagne agricole = millésime RPG (vérité terrain)
MILLESIME       = ANNEE_REFERENCE
REGION_CODE     = "R28"                # Normandie (code région INSEE 28)
BASE            = "PARCELLES_AGRICOLES_CONSTATEES"   # base « RPG Parcelles »
CRS_SOURCE      = "EPSG:2154"          # Lambert-93 (métropole)

# Fenêtre d'observation dérivée (sept N-1 -> déc N, avec RPG = N) :
FENETRE_DEBUT = date(ANNEE_REFERENCE - 1, 9, 1)
FENETRE_FIN   = date(ANNEE_REFERENCE, 12, 31)
# -> Les acquisitions Sentinel-2 seront ÉCHANTILLONNÉES dans cette fenêtre
#    (le satellite acquiert en continu ; sélection hors périmètre de ce notebook).

# ── Arborescence locale, ancrée sur la racine du dépôt (schéma brut) ──────
DATA_RAW = DATA_DIR / "raw" / "rpg" / str(MILLESIME) / REGION_CODE
DATA_RAW.mkdir(parents=True, exist_ok=True)

ARCHIVE_PATTERNS = ("*.7z", "*.zip")   # motif(s) de l'archive régionale RPG
SOURCE_URL       = ""                   # FACULTATIF — vide = téléchargement manuel

# ── Couche cible RPG et AOI ────────────────────────────────────────────────
GPKG         = next(DATA_RAW.rglob("RPG_Parcelles.gpkg"), None)
COUCHE_CIBLE = "RPG_Parcelles"
AOI_GEOJSON  = DATA_DIR / "vector" / "aoi" / "aoi_seinecrops.geojson"

# ── Connexion PostGIS (section 3+) ────────────────────────────────────────
load_dotenv(ROOT / ".env")
PG_PARAMS = {
    "host":     os.getenv("PG_HOST", "localhost"),
    "port":     int(os.getenv("PG_PORT", 5432)),
    "dbname":   os.getenv("PG_DBNAME", "seinecrops"),
    "user":     os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD"),
}
PSQL_BIN = r"C:\Program Files\PostgreSQL\18\bin\psql.exe"

print(f"Racine       : {ROOT}")
print(f"Référence    : {ANNEE_REFERENCE}  (RPG {MILLESIME}, {REGION_CODE})")
print(f"Fenêtre obs. : {FENETRE_DEBUT} -> {FENETRE_FIN}")
print(f"Dépôt local  : {DATA_RAW}")
print(f"GeoPackage   : {GPKG if GPKG else '(pas encore décompressé)'}")
print(f"AOI          : {AOI_GEOJSON}  ({'existe' if AOI_GEOJSON.exists() else 'MANQUANT'})")
print(f"Base cible   : {PG_PARAMS['user']}@{PG_PARAMS['host']}:{PG_PARAMS['port']}/{PG_PARAMS['dbname']}")

---

## 1 · Récupération du millésime

> **Objectif** — Identifier sans ambiguïté la source RPG retenue, la récupérer localement, 
> et en figer l'empreinte pour que la chaîne soit reproductible à l'identique.

**Offre 2024.** À partir du millésime **2024**, l'offre RPG passe d'une base unique à **sept bases 
thématiques** (huit avec les îlots, selon le descriptif de contenu v3.0), pour se conformer au règlement 
d'exécution européen 2023/138 sur les jeux de données de forte valeur. Pour SeineCrops, on retient 
**RPG Parcelles** (`PARCELLES_AGRICOLES_CONSTATEES`) : continuité directe du RPG historique 
*parcelle × culture* qui alimente la classification. (RPG PAC n'est qu'une agrégation en 4 grandes 
catégories — insuffisant pour notre usage.)

**Sens de la dépendance.** C'est la **campagne agricole / période d'observation** qui est choisie en 
premier (cf. cadrage). De cette période découlent (i) la **sélection** des acquisitions Sentinel-2 — 
échantillonnées *dans* la fenêtre, le satellite acquérant en continu — et (ii) le **millésime RPG** de 
référence, aligné sur cette même période (RPG N+1). Le millésime n'est donc **pas** « le dernier 
publié » : c'est celui qui correspond à la campagne étudiée.

**Pourquoi épingler plutôt qu'auto-sélectionner.** Un projet auditable doit rejouer la chaîne sur 
*exactement* la même donnée. On fixe donc le millésime explicitement (et son empreinte SHA-256 en 1.2). 
Le script **interroge** néanmoins les millésimes disponibles — pour **valider** le choix et alerter s'il 
n'est pas encore publié — sans jamais le décider à la volée.

| Élément | Valeur |
|---|---|
| Produit | Registre parcellaire graphique (RPG) |
| Base retenue | **RPG Parcelles** — *parcelles agricoles constatées* |
| Millésime | défini par `ANNEE_REFERENCE` (campagne de référence) — ex. 2024, arrêté au 1ᵉʳ janv. N+1 ; disponibilité validée en ligne |
| Emprise | Normandie — unité régionale **R28** |
| Producteur / diffuseur | ASP (gestion) · IGN (production, diffusion) · MASA (cadre réglementaire) |
| Licence | **Licence Ouverte / Etalab 2.0** |
| CRS source | Lambert-93 (EPSG:2154) |
| Documentation | `DC_DL_RPG_3-0.pdf` (descriptif) · `SE_RPG.pdf` (suivi des évolutions) |

Portails : 
[page produit RPG (cartes.gouv.fr)](https://cartes.gouv.fr/rechercher-une-donnee/dataset/IGNF_RPG) · 
[Géoservices IGN](https://geoservices.ign.fr/rpg) · 
[data.gouv.fr](https://www.data.gouv.fr/datasets/rpg).

#### Disponibilité du millésime (validation, pas auto-sélection)

On interroge le service **WFS** de la Géoplateforme et on extrait l'année **uniquement des noms de 
couches RPG** (balayer tout le `GetCapabilities` sur-détecterait des années d'autres jeux « agriculture » 
ou des libellés « édition 2026 »). Rappel métier : **le RPG d'une année N n'est diffusé qu'au 4ᵉ trimestre 
N+1** — le millésime le plus récent est donc en retard d'environ un an sur l'année courante. Les flux 
reflètent la disponibilité *en streaming*, qui peut différer un peu de l'**archive téléchargeable** : 
garde-fou, pas garantie. La fonction est tolérante aux pannes réseau.

In [ ]:
WFS_CAPS = ("https://data.geopf.fr/wfs/ows"
            "?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetCapabilities")


def _local(tag: str) -> str:
    """Nom local d'une balise XML, sans préfixe de namespace."""
    return tag.rsplit("}", 1)[-1]


def couches_rpg_wfs(url: str = WFS_CAPS, timeout: int = 60) -> list[tuple[int, str]]:
    """Couches RPG servies par le WFS Géoplateforme : liste de (millésime, nom_couche).

    Filtre strict : seuls les FeatureType dont le PRÉFIXE (avant ":") commence
    par "RPG." ou "IGNF_RPG_" sont retenus, et l'année n'est extraite QUE de ce
    préfixe — jamais du suffixe, qui peut contenir des dates parasites
    (ex. "surfaces_2024_zdh_20250621", ou une couche tierce comme
    "rpg_metrolyon_..._02-04-2026_wfs" qui n'a rien à voir avec le RPG national).

    Diagnostic seulement : ne décide jamais du millésime à charger.
    """
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            data = resp.read()
    except Exception as exc:  # noqa: BLE001
        print(f"Découverte indisponible ({exc!r}) — on garde le millésime pinné.")
        return []
    try:
        root = ET.fromstring(data)
    except ET.ParseError as exc:
        print(f"GetCapabilities illisible ({exc!r}) — on garde le millésime pinné.")
        return []

    res: list[tuple[int, str]] = []
    for ft in root.iter():
        if _local(ft.tag) != "FeatureType":
            continue
        nom = next((c.text for c in ft if _local(c.tag) == "Name" and c.text), None)
        if not nom or not re.match(r"(RPG\.|IGNF_RPG_)", nom, re.I):
            continue
        # N'extraire l'année QUE du préfixe (avant ":") :
        # les suffixes contiennent des dates parasites (ex. "20250621", "02-04-2026").
        prefixe = nom.split(":")[0]
        for a in re.findall(r"(20\d{2})", prefixe):
            if 2015 <= int(a) <= date.today().year + 1:
                res.append((int(a), nom))
    return sorted(set(res))


couches = couches_rpg_wfs()
dispo   = sorted({an for an, _ in couches})
if couches:
    print("Couches RPG vues dans le flux WFS (millésime : nom de couche) :")
    for an, nom in couches:
        print(f"  {an} : {nom}")
    print()
    recent = max(dispo)
    print(f"Millésimes RPG en flux : {dispo}")
    if MILLESIME in dispo:
        note = "le plus récent" if MILLESIME == recent else f"le plus récent est {recent}"
        print(f"OK  millésime {MILLESIME} servi en flux ({note}).")
    else:
        print(f"ATTENTION  millésime {MILLESIME} non vu dans les flux — plus récent vu : {recent}.")
else:
    print(f"Validation en ligne ignorée — millésime retenu : {MILLESIME}.")
print("Note : un flux peut devancer l'archive téléchargeable (placeholders, nommage v3.0) ;")
print("       la disponibilité réelle est tranchée au téléchargement en 1.2.")

In [ ]:
# ── Fiche de traçabilité ─────────────────────────────────────────────────
source = {
    "produit": "Registre parcellaire graphique (RPG)",
    "base": BASE,                       # RPG Parcelles — parcelles agricoles constatées
    "annee_reference": ANNEE_REFERENCE,
    "millesime": MILLESIME,
    "fenetre_observation": [FENETRE_DEBUT.isoformat(), FENETRE_FIN.isoformat()],
    "emprise_administrative": "Normandie (R28)",
    "producteur_diffuseur": "IGN (prod./diff.), ASP (gestion), MASA (cadre réglementaire)",
    "portails": [
        "https://cartes.gouv.fr/rechercher-une-donnee/dataset/IGNF_RPG",
        "https://geoservices.ign.fr/rpg",
        "https://www.data.gouv.fr/datasets/rpg",
    ],
    "licence": "Licence Ouverte / Etalab 2.0",
    "crs_source": CRS_SOURCE,
    "documentation": {
        "descriptif_contenu": "https://data.geopf.fr/annexes/ressources/documentation/DC_DL_RPG_3-0.pdf",
        "suivi_evolutions":  "https://data.geopf.fr/annexes/ressources/documentation/SE_RPG.pdf",
    },
    "millesimes_disponibles_wfs": dispo,
    "date_recuperation": date.today().isoformat(),
    "archive": None,                    # renseigné en 1.2 (détection auto)
    "url_archive": SOURCE_URL or None,
    "sha256": None,                     # renseigné en 1.2
    "statut": "init",                   # init -> archive_recuperee / archive_absente
    "note_offre_2024": (
        "Depuis le millésime 2024, l'offre RPG comporte 7 bases thématiques "
        "(8 avec les îlots, selon le descriptif v3.0). On retient RPG Parcelles, "
        "continuité du RPG historique parcelle x culture."
    ),
}

print(json.dumps(source, indent=2, ensure_ascii=False))

### 1.2 · Récupération et empreinte

Les archives régionales RPG sont volumineuses ; le téléchargement direct (`SOURCE_URL`) est 
**facultatif** — le plus souvent on télécharge manuellement depuis la page produit et on **dépose 
l'archive dans `DATA_RAW`**, où elle est **détectée automatiquement** (pas de nom de fichier à saisir). 
On calcule alors son **empreinte SHA-256**, qui épingle le millésime exact et complète la fiche de 
traçabilité. Tant qu'aucune archive n'est présente, le `statut` de la fiche le signale explicitement.

In [ ]:
def sha256sum(path: Path, chunk: int = 1 << 20) -> str:
    """Empreinte SHA-256 d'un fichier (pinning du millésime exact)."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


archive_path = None

# Téléchargement direct optionnel (si lien stable fourni et fichier encore absent).
if SOURCE_URL:
    cible = DATA_RAW / Path(SOURCE_URL).name
    if not cible.exists():
        print(f"Téléchargement -> {cible} ...")
        urllib.request.urlretrieve(SOURCE_URL, cible)  # noqa: S310
        print("Terminé.")

# Détection automatique de l'archive déposée dans DATA_RAW.
candidats = sorted({p for pat in ARCHIVE_PATTERNS for p in DATA_RAW.glob(pat)})

if len(candidats) == 1:
    archive_path = candidats[0]
    digest = sha256sum(archive_path)
    source["archive"] = archive_path.name
    source["sha256"]  = digest
    source["statut"]  = "archive_recuperee"
    print(f"OK  archive : {archive_path.name}  ({archive_path.stat().st_size / 1e6:.1f} Mo)")
    print(f"    SHA-256 : {digest}")
elif not candidats:
    source["statut"] = "archive_absente"
    print("Aucune archive détectée dans", DATA_RAW.resolve())
    print("  -> Page produit RPG : Normandie (R28), base RPG Parcelles, millésime cible ;")
    print("     déposer l'archive (.7z / .zip) dans ce dossier, puis ré-exécuter cette section.")
else:
    source["statut"] = "archive_ambigue"
    print("Plusieurs archives candidates — préciser ARCHIVE_PATTERNS :")
    for c in candidats:
        print("   -", c.name)

In [ ]:
# Persistance de la fiche de traçabilité (versionnée à côté du notebook).
source_file = DATA_RAW / "SOURCE.json"
source_file.write_text(json.dumps(source, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Fiche écrite : {source_file}  (statut : {source['statut']})")
if source["sha256"] is None:
    print("  Note : empreinte non calculée — fiche incomplète tant que l'archive n'est pas récupérée.")

### 1.3 · Référentiels associés

La donnée RPG porte des **codes** de culture, pas des libellés. On réserve ici les emplacements ; 
la récupération effective et la vérification du schéma se feront en **section 2 (reconnaissance)**, 
une fois l'archive décompressée.

In [ ]:
REF_DIR = DATA_RAW.parent / "_referentiels"
REF_DIR.mkdir(parents=True, exist_ok=True)

referentiels = {
    "codes_cultures": REF_DIR / f"codes_cultures_{MILLESIME}.csv",
    "descriptif_contenu": REF_DIR / "DC_DL_RPG_3-0.pdf",
}

for nom, chemin in referentiels.items():
    etat = "présent" if chemin.exists() else "à récupérer"
    print(f"{nom:20s} : {chemin.name}  ({etat})")

---

## 2 · Reconnaissance

> **Objectif** — Inventorier le contenu du GeoPackage en lecture seule : couches disponibles, 
> schéma attributaire, type de géométrie, CRS et volume. Aucune donnée n'est chargée en mémoire 
> complète au-delà de ce qui est nécessaire pour les statistiques ; on se donne une image fidèle 
> de la livraison avant toute transformation.

Cette section est purement **exploratoire** : elle ne modifie ni la base PostGIS ni l'archive. 
Elle produit en revanche un **rapport de reconnaissance** (`RECON.json`) versionné à côté de 
`SOURCE.json`.

### 2.1 · Localisation et décompression

On localise le `.gpkg` dans `DATA_RAW` (`rglob`, car `py7zr` extrait dans une sous-arborescence). 
S'il est absent mais qu'une archive `.7z` est présente, on la décompresse automatiquement.

In [ ]:
gpkg_cible = list(DATA_RAW.rglob("RPG_Parcelles.gpkg"))

if not gpkg_cible:
    archives = sorted(DATA_RAW.glob("*.7z"))
    if not archives:
        raise FileNotFoundError(
            f"Ni RPG_Parcelles.gpkg ni .7z trouvé sous {DATA_RAW}\n"
            "-> Vérifier que DATA_RAW pointe sur le bon dossier."
        )
    import py7zr
    archive = archives[0]
    print(f"Décompression de {archive.name} ({archive.stat().st_size / 1e9:.2f} Go)…")
    with py7zr.SevenZipFile(archive, mode="r") as z:
        z.extractall(path=DATA_RAW)
    print("Terminé.")
    gpkg_cible = list(DATA_RAW.rglob("RPG_Parcelles.gpkg"))

if not gpkg_cible:
    raise FileNotFoundError("RPG_Parcelles.gpkg introuvable après décompression.")

GPKG = gpkg_cible[0]
print(f"OK  GeoPackage : {GPKG.name}")
print(f"    Chemin     : {GPKG.relative_to(DATA_RAW)}")
print(f"    Taille     : {GPKG.stat().st_size / 1e9:.2f} Go")

print()
print("Autres bases présentes dans la livraison :")
tous_gpkg = sorted(DATA_RAW.rglob("RPG_*.gpkg"))
for p in tous_gpkg:
    if p != GPKG:
        print(f"  {p.name}")

### 2.2 · Inventaire des couches

`fiona.listlayers` liste les couches du fichier. La v3.0 livre une couche par fichier 
GeoPackage (et non plusieurs couches dans un fichier unique, comme en v2.0) — le nom réel 
de la couche cible (`RPG_Parcelles`, pas `parcelles_graphiques`) en découle.

In [ ]:
couches_dispo = fiona.listlayers(GPKG)
print(f"{len(couches_dispo)} couche(s) dans {GPKG.name} :\n")

inventaire = []
for nom in couches_dispo:
    with fiona.open(GPKG, layer=nom) as src:
        entree = {
            "couche":    nom,
            "geom":      src.schema["geometry"] if src.schema["geometry"] != "None" else "—",
            "epsg":      src.crs.to_epsg() if src.crs else None,
            "n_objets":  len(src),
            "attributs": list(src.schema["properties"].keys()),
        }
        inventaire.append(entree)
        geom_str = entree["geom"] if entree["geom"] != "—" else "(table)"
        print(f"  {nom:<40s}  {geom_str:<20s}  EPSG:{entree['epsg']}"
              f"  {entree['n_objets']:>8,} objets")

# La couche cible est la première (et probablement unique) couche du fichier
COUCHE_CIBLE = couches_dispo[0]
print(f"\nCouche cible retenue : {COUCHE_CIBLE!r}")

### 2.3 · Schéma attributaire de `RPG_Parcelles`

C'est la couche cible de SeineCrops. Les attributs v3.0 sont en **minuscules** 
(`id_parcel`, `surf_parc`, `code_cultu`, `code_group`, `culture_d1`, `culture_d2`), 
contrairement à la documentation v2.0 qui les présente en majuscules. Un nouvel attribut 
apparaît en v3.0 : `cat_cult_p` (catégorie PAC : TA, CP, PP, SB).

In [ ]:
with fiona.open(GPKG, layer=COUCHE_CIBLE) as src:
    schema   = src.schema
    crs      = src.crs
    n_objets = len(src)

print(f"Couche    : {COUCHE_CIBLE}")
print(f"Géométrie : {schema['geometry']}")
print(f"CRS       : EPSG:{crs.to_epsg()}")
print(f"Objets    : {n_objets:,}")
print()
print("Attributs :")
attrs_attendus = {"id_parcel", "surf_parc", "code_cultu", "code_group", "culture_d1", "culture_d2"}
for attr, dtype in schema["properties"].items():
    flag = "  ← attendu" if attr in attrs_attendus else "  ← nouveauté v3.0" if attr == "cat_cult_p" else ""
    print(f"  {attr:<20s}  {dtype}{flag}")

attrs_manquants = attrs_attendus - set(schema["properties"])
if attrs_manquants:
    print(f"\nATTENTION  attributs attendus non trouvés : {attrs_manquants}")
else:
    print("\nOK  tous les attributs attendus sont présents.")

### 2.4 · Aperçu des données et distributions

**Attention au biais d'échantillonnage** : `max_features` charge les *premières lignes* du fichier, 
pas un échantillon aléatoire. Sur ce GeoPackage, les premières lignes correspondent à de très 
petites parcelles (tri interne probable par `id_parcel` croissant) — un `max_features=50000` 
donnerait des statistiques de surface fortement biaisées. On charge donc la couche **complète** 
(528 950 parcelles, 0,36 Go — largement gérable en mémoire) pour des statistiques fiables.

In [ ]:
df_sample = pyogrio.read_dataframe(GPKG, layer=COUCHE_CIBLE)

print(f"Parcelles chargées : {len(df_sample):,}")
print(f"Colonnes           : {list(df_sample.columns)}")
print()
df_sample.head(5)

In [ ]:
# ── Distribution des cultures principales ────────────────────────────────
top_cultures = (
    df_sample["code_cultu"]
    .value_counts()
    .head(20)
    .rename_axis("code_cultu")
    .reset_index(name="n_parcelles")
)
top_cultures["pct"] = (top_cultures["n_parcelles"] / len(df_sample) * 100).round(1)
print("Top 20 cultures :")
print(top_cultures.to_string(index=False))

In [ ]:
# ── Distribution des surfaces ─────────────────────────────────────────────
stats_surf = df_sample["surf_parc"].describe().round(3)
print("Surface (ha) — statistiques descriptives (528 950 parcelles, Normandie entière) :")
print(stats_surf.to_string())
print()

grandes = df_sample[df_sample["surf_parc"] > 500]
if not grandes.empty:
    print(f"ATTENTION  {len(grandes)} parcelle(s) > 500 ha.")
else:
    print("OK  aucune parcelle > 500 ha.")

### 2.5 · Emprise spatiale

`pyogrio.read_info` donne l'emprise totale sans charger les géométries — utile pour confirmer 
la couverture géographique avant tout traitement plus lourd.

In [ ]:
info = pyogrio.read_info(GPKG, layer=COUCHE_CIBLE)
bbox = info["total_bounds"]

print("Emprise (EPSG:2154, Lambert-93) :")
print(f"  xmin (W) : {bbox[0]:,.0f} m")
print(f"  ymin (S) : {bbox[1]:,.0f} m")
print(f"  xmax (E) : {bbox[2]:,.0f} m")
print(f"  ymax (N) : {bbox[3]:,.0f} m")
print()
print("Cohérence avec la Normandie (Lambert-93) :")
checks = [
    ("xmin > 300 000", bbox[0] > 300_000),
    ("xmax < 750 000", bbox[2] < 750_000),
    ("ymin > 6 700 000", bbox[1] > 6_700_000),
    ("ymax < 7 000 000", bbox[3] < 7_000_000),
]
for label, ok in checks:
    print(f"  {'OK ' if ok else 'KO '} {label}")

### 2.6 · Autres bases de la livraison et table de référence

La v3.0 livre les bases thématiques dans des **fichiers GeoPackage séparés** (RPG_Parcelles, 
RPG_Ilots, RPG_PAC, RPG_PP, RPG_BIO, RPG_IAE, RPG_SNA, RPG_ZDH). On inventorie ici les 
couches disponibles dans chaque fichier.

La table de référence `codes_cultures` (`code_cultu → libellé`) est **absente de l'archive 
régionale v3.0** — elle n'est plus livrée localement. On la récupère via le flux **WFS 
Géoplateforme** (`RPG.2024:codes_cultures`), qui expose les 147 codes nationaux. C'est l'un 
des rares cas où la lecture en ligne est justifiée : table légère, stable, sans géométrie, 
qui n'a pas besoin d'être épinglée par SHA-256.

In [ ]:
# Inventaire des bases de la livraison
print("Bases disponibles dans la livraison :")
for p in tous_gpkg:
    couches = fiona.listlayers(p)
    print(f"  {p.name:<25s}  {len(couches)} couche(s) : {couches}")

# codes_cultures absent de l'archive v3.0 — récupération via WFS Géoplateforme
print()
print("Table codes_cultures absente de l'archive — récupération via WFS…")
url_codes = (
    "https://data.geopf.fr/wfs/ows"
    "?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature"
    "&TYPENAMES=RPG.2024:codes_cultures"
    "&OUTPUTFORMAT=application%2Fjson"
)
with urllib.request.urlopen(url_codes, timeout=60) as r:
    data_codes = json.loads(r.read())

df_codes = pd.DataFrame([f["properties"] for f in data_codes["features"]])
print(f"OK  {len(df_codes)} codes  |  colonnes : {list(df_codes.columns)}")
print()
print(df_codes.head(10).to_string(index=False))

dest_codes = REF_DIR / f"codes_cultures_{MILLESIME}.csv"
df_codes.to_csv(dest_codes, index=False, encoding="utf-8")
print(f"\nExporté : {dest_codes}")

### 2.7 · Rapport de reconnaissance

On consolide les résultats en un `RECON.json` versionné, qui complète `SOURCE.json`.

In [ ]:
recon = {
    "gpkg": GPKG.name,
    "chemin_relatif": str(GPKG.relative_to(DATA_RAW)),
    "taille_go": round(GPKG.stat().st_size / 1e9, 3),
    "date_reconnaissance": datetime.date.today().isoformat(),
    "couches": inventaire,
    "RPG_Parcelles": {
        "n_objets":  n_objets,
        "geometrie": schema["geometry"],
        "epsg":      crs.to_epsg(),
        "attributs_livres": list(schema["properties"].keys()),
        "attributs_attendus_v3": [
            "id_parcel", "surf_parc", "code_cultu", "code_group",
            "culture_d1", "culture_d2", "cat_cult_p",
        ],
        "note_cat_cult_p": "Nouveauté v3.0 — catégorie PAC (TA, CP, PP, SB)",
        "bbox_lamb93": {
            "xmin": round(bbox[0]),
            "ymin": round(bbox[1]),
            "xmax": round(bbox[2]),
            "ymax": round(bbox[3]),
        },
    },
    "surf_parc_stats_completes": stats_surf.to_dict(),
    "top20_cultures": top_cultures.to_dict(orient="records"),
    "codes_cultures": {
        "source": "WFS Géoplateforme — RPG.2024:codes_cultures",
        "url": url_codes,
        "n_codes": len(df_codes),
        "note": "Absent de l'archive régionale v3.0 — récupéré via WFS.",
        "export": f"codes_cultures_{MILLESIME}.csv",
    },
    "autres_bases_livraison": [p.name for p in tous_gpkg if p != GPKG],
}

dest_recon = DATA_RAW / "RECON.json"
dest_recon.write_text(json.dumps(recon, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Rapport écrit : {dest_recon}")

---

## 3 · Préparation de PostGIS

> **Objectif** — Vérifier la connexion à la base, confirmer que PostGIS est actif, et confirmer 
> la présence des schémas `raw` (données source non modifiées) et `derived` (transformations).

La création de la base elle-même (`CREATE DATABASE seinecrops`) et l'activation de l'extension 
(`CREATE EXTENSION postgis`) sont faites une fois via `src/db/init.sql`, hors notebook — ce sont 
des opérations d'infrastructure, pas d'exploration. Cette section ne fait que **vérifier** l'état 
et créer les schémas si besoin.

### 3.1 · Connexion et vérification PostGIS

In [ ]:
with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:

        cur.execute("SELECT version();")
        print("PostgreSQL :", cur.fetchone()[0].split(",")[0])

        cur.execute("SELECT PostGIS_version();")
        print("PostGIS    :", cur.fetchone()[0])

        cur.execute("""
            SELECT schema_name
            FROM information_schema.schemata
            WHERE schema_name IN ('raw', 'derived')
            ORDER BY schema_name;
        """)
        schemas = [r[0] for r in cur.fetchall()]
        print()
        for s in ("raw", "derived"):
            print(f"  {'OK ' if s in schemas else 'MANQUANT'} schéma {s!r}")

In [ ]:
# Rapport de connexion, versionné à côté de SOURCE.json et RECON.json
with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        pg_version = cur.fetchone()[0].split(",")[0]
        cur.execute("SELECT PostGIS_version();")
        postgis_version = cur.fetchone()[0]

db_report = {
    "date": datetime.date.today().isoformat(),
    "host": PG_PARAMS["host"],
    "port": PG_PARAMS["port"],
    "dbname": PG_PARAMS["dbname"],
    "user": PG_PARAMS["user"],
    "postgresql": pg_version,
    "postgis": postgis_version,
    "schemas": ["raw", "derived"],
}

dest_db = DATA_RAW / "DB.json"
dest_db.write_text(json.dumps(db_report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Rapport écrit : {dest_db}")

---
*Fin de la section 3.* La section 4 filtrera `RPG_Parcelles` sur l'AOI 
(Caux + Neubourg, cross-Seine) avant chargement dans le schéma `raw`.

## 4 · Découpe spatiale sur l'AOI

> **Objectif** — Charger `RPG_Parcelles` intégralement dans le schéma `raw` (brut, non modifié), 
> puis produire dans le schéma `derived` une version **filtrée sur l'AOI** (Pays de Caux + plateau 
> du Neubourg, cross-Seine). Le filtre se fait **côté PostGIS** via `ST_Intersects`, pas en mémoire 
> avec GeoPandas — choix pédagogique : on manipule la base spatiale directement, dans l'esprit 
> d'une chaîne opérationnelle où PostGIS porte la logique, pas le notebook.

**Choix : parcelles entières, pas de découpe à la frontière.** Une parcelle à cheval sur la 
limite de l'AOI est conservée **intégralement** plutôt que tronquée. Une parcelle découpée 
perdrait sa cohérence phénologique : la signature spectrale d'un fragment de parcelle mélange 
des effets de bord qui n'ont pas de sens pour la classification de culture. On retient donc 
`ST_Intersects` comme filtre, sans appliquer `ST_Intersection`.

**Note technique — chargement via PGDUMP.** `ogr2ogr` et le driver `PostgreSQL` de `pyogrio` ne 
sont pas disponibles dans cet environnement (GDAL embarqué dans les wheels `fiona`/`pyogrio`, sans 
binaire CLI ni driver PostgreSQL compilé). Le chargement passe donc par le driver **PGDUMP** 
(export `.sql` au format `COPY` natif), chargé ensuite via `psql` en `subprocess` — `psql` étant 
disponible (chemin explicite, le PATH du shell ne se propageant pas au kernel Jupyter). GDAL nomme 
par défaut la colonne géométrie `wkb_geometry` (pas `geom`) dans ce driver.

**Étapes :**

| # | Étape | Schéma cible |
|---|---|---|
| 4.1 | Chargement de `RPG_Parcelles` (528 950 parcelles, intégral) | `raw` |
| 4.2 | Chargement de l'AOI (`aoi_seinecrops.geojson`) | `raw` |
| 4.3 | Filtre via `ST_Intersects` (SQL, parcelles entières) | `derived` |
| 4.4 | Indexation spatiale (GIST) sur la table filtrée | `derived` |
| 4.5 | Contrôle visuel et statistiques du filtre | — |

**Note de séparation brut / dérivé.** `raw.rpg_parcelles` n'est jamais modifié après chargement : 
c'est la référence intégrale, reproductible depuis `SOURCE.json`. Toute transformation 
(filtre AOI, reprojection UTM 31N à venir) vit dans `derived`.

### 4.1 · Chargement de `RPG_Parcelles` dans `raw`

In [ ]:
print(f"Lecture de {COUCHE_CIBLE} ({GPKG.name})…")
df_full = pyogrio.read_dataframe(GPKG, layer=COUCHE_CIBLE)
n_objets = len(df_full)
print(f"OK  {n_objets:,} parcelles lues en mémoire.")

# Export au format PGDUMP (COPY natif PostgreSQL) — cf. note technique ci-dessus.
dump_path = DATA_RAW / "rpg_parcelles_raw.sql"

print("Export PGDUMP en cours…")
pyogrio.write_dataframe(
    df_full,
    str(dump_path),
    driver="PGDUMP",
    layer="rpg_parcelles",
    layer_options={"SCHEMA": "raw", "GEOM_TYPE": "geometry"},
)
print(f"OK  dump écrit : {dump_path}  ({dump_path.stat().st_size / 1e6:.1f} Mo)")

# Nettoyage : retirer le CREATE SCHEMA généré par GDAL — le schéma raw
# existe déjà (créé en section 3) ; le conserver ferait échouer le chargement
# avec ON_ERROR_STOP=1 et empêcherait toute instruction suivante de s'exécuter.
contenu = dump_path.read_text(encoding="utf-8")
contenu_nettoye = "\n".join(
    line for line in contenu.splitlines()
    if not line.strip().upper().startswith("CREATE SCHEMA")
)
dump_path.write_text(contenu_nettoye, encoding="utf-8")
print("OK  CREATE SCHEMA retiré du dump.")

In [ ]:
cmd_psql = [
    PSQL_BIN,
    "-U", PG_PARAMS["user"],
    "-h", PG_PARAMS["host"],
    "-p", str(PG_PARAMS["port"]),
    "-d", PG_PARAMS["dbname"],
    "-v", "ON_ERROR_STOP=1",
    "-f", str(dump_path),
]
env = {**os.environ, "PGPASSWORD": PG_PARAMS["password"]}

print("Chargement de raw.rpg_parcelles (peut prendre quelques minutes)…")
result = subprocess.run(cmd_psql, capture_output=True, text=True, env=env)

if result.returncode == 0:
    print("OK  chargement terminé.")
else:
    print("ERREUR psql :")
    print(result.stderr)

In [ ]:
# Vérification du chargement
with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM raw.rpg_parcelles;")
        n_charge = cur.fetchone()[0]
        cur.execute("SELECT ST_SRID(wkb_geometry) FROM raw.rpg_parcelles LIMIT 1;")
        srid = cur.fetchone()[0]

print(f"Parcelles chargées : {n_charge:,}")
print(f"SRID               : {srid}")
assert n_charge == n_objets, f"Attendu {n_objets:,}, trouvé {n_charge:,}"
assert srid == 2154, f"SRID inattendu : {srid}"
print("OK  cohérent avec la reconnaissance (section 2).")

### 4.1bis · QA géométrique de `raw.rpg_parcelles` (avant filtre)

**Pourquoi ici, et pas après le filtre.** La requête de filtre AOI (4.3) exclut les 
géométries invalides via `WHERE ST_IsValid(...)`. Si cette validation n'est faite **qu'après** 
le filtre, une parcelle invalide située dans l'AOI disparaît silencieusement de 
`derived.rpg_parcelles_aoi` sans qu'aucune trace n'explique l'effectif manquant — la QA en fin 
de notebook la découvrirait trop tard pour comprendre la cause.

On valide donc **`raw.rpg_parcelles` dans son ensemble**, avant le filtre — pas seulement le 
sous-ensemble qui tombe dans l'AOI. Coût accepté : on contrôle des parcelles qui ne seront 
peut-être jamais utilisées par SeineCrops, en échange d'une garantie que toute parcelle de 
l'AOI, valide ou réparée, *peut* entrer dans `derived` plutôt que d'être perdue sans trace.

La réparation via `ST_MakeValid` n'est appliquée **que si** des invalidités sont détectées, et 
uniquement aux lignes concernées.

In [ ]:
def qa_validite(schema_table: str, col_geom: str) -> int:
    """Compte les géométries invalides dans une table. Retourne le nombre trouvé."""
    with psycopg2.connect(**PG_PARAMS) as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                SELECT COUNT(*) FROM {schema_table}
                WHERE NOT ST_IsValid({col_geom});
            """)
            return cur.fetchone()[0]


def reparer_si_necessaire(schema_table: str, col_geom: str, n_invalides: int) -> int:
    """Répare les géométries invalides via ST_MakeValid si nécessaire.

    Ne touche pas à la table si aucune invalidité n'est détectée — la réparation
    n'est appliquée qu'aux lignes concernées, pas en bloc sur toute la table.
    Retourne le nombre de géométries effectivement réparées.
    """
    if n_invalides == 0:
        print(f"OK  {schema_table} — aucune réparation nécessaire.")
        return 0

    with psycopg2.connect(**PG_PARAMS) as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                UPDATE {schema_table}
                SET {col_geom} = ST_MakeValid({col_geom})
                WHERE NOT ST_IsValid({col_geom});
            """)
            n_reparees = cur.rowcount
            conn.commit()
    print(f"OK  {schema_table} — {n_reparees:,} géométrie(s) réparée(s) via ST_MakeValid.")
    return n_reparees


n_invalides_raw = qa_validite("raw.rpg_parcelles", "wkb_geometry")
print(f"Géométries invalides — raw.rpg_parcelles : {n_invalides_raw:,}  (sur {n_charge:,})")

n_reparees_raw = reparer_si_necessaire("raw.rpg_parcelles", "wkb_geometry", n_invalides_raw)

### 4.2 · Chargement de l'AOI dans `raw`

L'AOI (`aoi_seinecrops.geojson`, EPSG:4326) est chargée et reprojetée en EPSG:2154 au 
chargement, pour être comparable à `raw.rpg_parcelles`.

In [ ]:
print(f"Lecture de l'AOI ({AOI_GEOJSON.name})…")
aoi = gpd.read_file(AOI_GEOJSON)
print(f"OK  {len(aoi)} entité(s), CRS source : {aoi.crs}")

aoi_2154 = aoi.to_crs(epsg=2154)
print(f"Reprojeté en : {aoi_2154.crs}")

dump_aoi_path = DATA_RAW / "aoi_seinecrops_raw.sql"

pyogrio.write_dataframe(
    aoi_2154,
    str(dump_aoi_path),
    driver="PGDUMP",
    layer="aoi_seinecrops",
    layer_options={"SCHEMA": "raw", "GEOM_TYPE": "geometry"},
)
print(f"OK  dump écrit : {dump_aoi_path}")

contenu = dump_aoi_path.read_text(encoding="utf-8")
contenu_nettoye = "\n".join(
    line for line in contenu.splitlines()
    if not line.strip().upper().startswith("CREATE SCHEMA")
)
dump_aoi_path.write_text(contenu_nettoye, encoding="utf-8")
print("OK  CREATE SCHEMA retiré du dump.")

In [ ]:
cmd_psql_aoi = [
    PSQL_BIN,
    "-U", PG_PARAMS["user"],
    "-h", PG_PARAMS["host"],
    "-p", str(PG_PARAMS["port"]),
    "-d", PG_PARAMS["dbname"],
    "-v", "ON_ERROR_STOP=1",
    "-f", str(dump_aoi_path),
]

print("Chargement de raw.aoi_seinecrops…")
result = subprocess.run(cmd_psql_aoi, capture_output=True, text=True, env=env)

if result.returncode == 0:
    print("OK  chargement terminé.")
else:
    print("ERREUR psql :")
    print(result.stderr)

In [ ]:
with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT ST_SRID(wkb_geometry), ROUND((ST_Area(wkb_geometry) / 1e6)::numeric, 1)
            FROM raw.aoi_seinecrops;
        """)
        srid_aoi, surf_km2 = cur.fetchone()

print(f"SRID AOI    : {srid_aoi}")
print(f"Surface AOI : {surf_km2:,} km²")
assert srid_aoi == 2154
print("OK  AOI chargée et reprojetée.")

### 4.3 · Filtre via `ST_Intersects` (révisé)

On produit `derived.rpg_parcelles_aoi` : les parcelles **intersectant** l'AOI, conservées 
**entières**. La clause `WHERE ST_IsValid(...)` a été retirée — `raw.rpg_parcelles` a déjà été 
validée et réparée en 4.1bis, ce filtre serait donc redondant et masquerait une éventuelle 
régression silencieusement.

In [ ]:
sql_filtre = """
DROP TABLE IF EXISTS derived.rpg_parcelles_aoi;

CREATE TABLE derived.rpg_parcelles_aoi AS
SELECT
    p.id_parcel,
    p.surf_parc,
    p.code_cultu,
    p.code_group,
    p.culture_d1,
    p.culture_d2,
    p.cat_cult_p,
    p.wkb_geometry AS geom                          -- renommé pour la suite du notebook
FROM raw.rpg_parcelles AS p
JOIN raw.aoi_seinecrops AS a
    ON ST_Intersects(p.wkb_geometry, a.wkb_geometry);
"""

with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:
        print("Filtre en cours (ST_Intersects sur la Normandie entière)…")
        cur.execute(sql_filtre)
        conn.commit()

print("OK  derived.rpg_parcelles_aoi créée (parcelles entières, géométries déjà validées).")

### 4.4 · Indexation spatiale

Un index **GIST** sur la géométrie est indispensable pour toute requête spatiale ultérieure 
(jointures avec Sentinel-2, agrégations zonales).

In [ ]:
sql_index = """
CREATE INDEX IF NOT EXISTS idx_rpg_parcelles_aoi_geom
    ON derived.rpg_parcelles_aoi
    USING GIST (geom);

CREATE INDEX IF NOT EXISTS idx_rpg_parcelles_aoi_code_cultu
    ON derived.rpg_parcelles_aoi (code_cultu);
"""

with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:
        cur.execute(sql_index)
        conn.commit()

print("OK  index GIST (géométrie) et B-tree (code_cultu) créés.")

### 4.5 · Contrôle du filtre

On vérifie le volume, la surface totale et la cohérence géométrique du résultat.

In [ ]:
with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM derived.rpg_parcelles_aoi;")
        n_aoi = cur.fetchone()[0]

        cur.execute("""
            SELECT ROUND((SUM(surf_parc))::numeric, 1)
            FROM derived.rpg_parcelles_aoi;
        """)
        surf_totale_ha = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*) FROM derived.rpg_parcelles_aoi
            WHERE NOT ST_IsValid(geom);
        """)
        n_invalides = cur.fetchone()[0]

        cur.execute("""
            SELECT ROUND(ST_XMin(ext)::numeric), ROUND(ST_YMin(ext)::numeric),
                   ROUND(ST_XMax(ext)::numeric), ROUND(ST_YMax(ext)::numeric)
            FROM (SELECT ST_Extent(geom) AS ext FROM derived.rpg_parcelles_aoi) t;
        """)
        bbox_aoi = cur.fetchone()

print(f"Parcelles dans l\'AOI    : {n_aoi:,}  (sur {n_charge:,} en Normandie)")
print(f"Surface totale          : {surf_totale_ha:,} ha  ({surf_totale_ha / 100:.0f} km²)")
print(f"Géométries invalides    : {n_invalides}")
print(f"Emprise (Lambert-93)    : {bbox_aoi}")

if n_invalides:
    print("\nATTENTION  des géométries invalides subsistent — "
          "envisager ST_MakeValid en section 7 (QA).")
else:
    print("\nOK  toutes les géométries du filtre sont valides.")

## 5 · Validation et documentation

> **Objectif** — Clore la chaîne d'ingestion par un contrôle de cohérence post-filtre et un 
> rapport de clôture qui consolide la traçabilité de bout en bout : source, millésime, filtre 
> appliqué, référentiel des cultures.

**Ce qui est déjà couvert.** L'indexation spatiale (4.4) et la QA géométrique de `raw` 
(4.1bis, avant le filtre) sont faites en amont. `derived.rpg_parcelles_aoi` hérite directement 
de géométries déjà valides, puisque le filtre (4.3) ne recalcule aucune géométrie — un simple 
contrôle de cohérence suffit ici, pas une revalidation complète.

**Étapes :**

| # | Étape |
|---|---|
| 5.1 | Contrôle de cohérence post-filtre (derived hérite-t-elle bien de raw, sans fuite ?) |
| 5.2 | Assertions formalisées : CRS, volumes, index |
| 5.3 | Rapport de clôture (`INGESTION_REPORT.json`) |

### 5.1 · Contrôle de cohérence post-filtre

On vérifie que `derived.rpg_parcelles_aoi` n'a pas de géométrie invalide — attendu, puisqu'elle 
hérite de `raw` déjà validée — et qu'aucune parcelle de `derived` ne s'écarte de `raw` (pas de 
fuite, pas de duplication introduite par la jointure du filtre).

In [ ]:
n_invalides_aoi = qa_validite("derived.rpg_parcelles_aoi", "geom")
print(f"Géométries invalides — derived.rpg_parcelles_aoi : {n_invalides_aoi:,}")

if n_invalides_aoi:
    print("ATTENTION  des géométries invalides sont apparues malgré la QA amont (4.1bis) —")
    print("           vérifier si le filtre (4.3) a recalculé une géométrie par erreur.")
    n_reparees_aoi = reparer_si_necessaire("derived.rpg_parcelles_aoi", "geom", n_invalides_aoi)
else:
    n_reparees_aoi = 0
    print("OK  conforme à l'attendu (derived hérite des géométries déjà validées en 4.1bis).")

### 5.2 · Assertions formalisées

On rassemble tous les contrôles dans un seul bloc

In [ ]:
with psycopg2.connect(**PG_PARAMS) as conn:
    with conn.cursor() as cur:

        # CRS cohérent sur les trois tables
        cur.execute("SELECT ST_SRID(wkb_geometry) FROM raw.rpg_parcelles LIMIT 1;")
        srid_raw = cur.fetchone()[0]
        cur.execute("SELECT ST_SRID(wkb_geometry) FROM raw.aoi_seinecrops LIMIT 1;")
        srid_aoi_brut = cur.fetchone()[0]
        cur.execute("SELECT ST_SRID(geom) FROM derived.rpg_parcelles_aoi LIMIT 1;")
        srid_derived = cur.fetchone()[0]

        # Volumes
        cur.execute("SELECT COUNT(*) FROM raw.rpg_parcelles;")
        n_raw = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM derived.rpg_parcelles_aoi;")
        n_derived = cur.fetchone()[0]

        # Cohérence du filtre : derived doit être un sous-ensemble strict de raw,
        # et chaque id_parcel de derived doit exister dans raw (pas de fuite/duplication)
        cur.execute("""
            SELECT COUNT(*) FROM derived.rpg_parcelles_aoi d
            WHERE NOT EXISTS (
                SELECT 1 FROM raw.rpg_parcelles r WHERE r.id_parcel = d.id_parcel
            );
        """)
        n_orphelines = cur.fetchone()[0]

        # Index présents
        cur.execute("""
            SELECT indexname FROM pg_indexes
            WHERE schemaname = 'derived' AND tablename = 'rpg_parcelles_aoi';
        """)
        index_presents = [r[0] for r in cur.fetchall()]

assert srid_raw == 2154, f"CRS raw.rpg_parcelles inattendu : {srid_raw}"
assert srid_aoi_brut == 2154, f"CRS raw.aoi_seinecrops inattendu : {srid_aoi_brut}"
assert srid_derived == 2154, f"CRS derived.rpg_parcelles_aoi inattendu : {srid_derived}"
print("OK  CRS cohérent (EPSG:2154) sur les 3 tables.")

assert 0 < n_derived < n_raw, f"Volume incohérent : derived={n_derived:,}, raw={n_raw:,}"
print(f"OK  derived ({n_derived:,}) est un sous-ensemble strict de raw ({n_raw:,}).")

assert n_orphelines == 0, f"{n_orphelines} parcelle(s) de derived sans correspondance dans raw"
print("OK  aucune parcelle orpheline — derived est bien un filtre de raw, sans fuite.")

assert n_invalides_aoi == 0, f"{n_invalides_aoi} géométrie(s) invalide(s) dans derived"
print("OK  toutes les géométries de derived sont valides.")

assert any("geom" in idx for idx in index_presents), "Index GIST manquant"
assert any("code_cultu" in idx for idx in index_presents), "Index code_cultu manquant"
print(f"OK  index présents : {index_presents}")

print("\nToutes les assertions sont passées — derived.rpg_parcelles_aoi est validée.")

### 5.3 · Rapport de clôture

On consolide `SOURCE.json`, `RECON.json` et `DB.json` en un seul document de synthèse — le 
point d'entrée pour quelqu'un qui découvre la chaîne d'ingestion sans avoir suivi chaque 
section. Il documente la source, le filtre appliqué, et référence le dictionnaire des cultures.

In [ ]:
def charger_si_existe(path: Path) -> dict | None:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


source_doc = charger_si_existe(DATA_RAW / "SOURCE.json")
recon_doc  = charger_si_existe(DATA_RAW / "RECON.json")
db_doc     = charger_si_existe(DATA_RAW / "DB.json")

ingestion_report = {
    "date_cloture": datetime.date.today().isoformat(),
    "millesime": MILLESIME,
    "region": REGION_CODE,
    "source": {
        "archive": source_doc.get("archive") if source_doc else None,
        "sha256": source_doc.get("sha256") if source_doc else None,
        "licence": source_doc.get("licence") if source_doc else None,
        "date_recuperation": source_doc.get("date_recuperation") if source_doc else None,
    },
    "reconnaissance": {
        "n_objets_normandie": recon_doc.get("RPG_Parcelles", {}).get("n_objets") if recon_doc else None,
        "schema_attributaire": recon_doc.get("RPG_Parcelles", {}).get("attributs_livres") if recon_doc else None,
    },
    "base_postgis": {
        "host": db_doc.get("host") if db_doc else None,
        "dbname": db_doc.get("dbname") if db_doc else None,
        "postgresql": db_doc.get("postgresql") if db_doc else None,
        "postgis": db_doc.get("postgis") if db_doc else None,
    },
    "filtre_aoi": {
        "aoi_fichier": str(AOI_GEOJSON.relative_to(ROOT)),
        "table_resultat": "derived.rpg_parcelles_aoi",
        "methode": "ST_Intersects (parcelles entières, pas de découpe à la frontière)",
        "n_parcelles_normandie": n_raw,
        "n_parcelles_aoi": n_derived,
        "surface_totale_ha": float(surf_totale_ha),
    },
    "qa": {
        "geometries_invalides_raw_avant_filtre": n_invalides_raw,
        "geometries_reparees_raw_avant_filtre": n_reparees_raw,
        "geometries_invalides_derived_apres_filtre": n_invalides_aoi,
        "geometries_reparees_derived_apres_filtre": n_reparees_aoi,
        "parcelles_orphelines": n_orphelines,
        "index_presents": index_presents,
        "note_ordre_qa": (
            "QA géométrique appliquée à raw AVANT le filtre AOI (4.1bis), pas après : "
            "une parcelle invalide dans l'AOI doit être réparée ou explicitement tracée, "
            "pas silencieusement exclue par un WHERE ST_IsValid dans la requête de filtre."
        ),
    },
    "referentiel_cultures": {
        "fichier": str((DATA_RAW.parent / "_referentiels" / f"codes_cultures_{MILLESIME}.csv").relative_to(ROOT)),
        "n_codes": len(df_codes) if "df_codes" in dir() else None,
        "source": "WFS Géoplateforme — RPG.{}:codes_cultures".format(MILLESIME),
    },
}

dest_report = DATA_RAW / "INGESTION_REPORT.json"
dest_report.write_text(json.dumps(ingestion_report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Rapport de clôture écrit : {dest_report}")
print()
print(json.dumps(ingestion_report, indent=2, ensure_ascii=False))

---
*Fin de l'ingestion RPG.* `derived.rpg_parcelles_aoi` (80 689 parcelles, AOI Pays de Caux + 
Neubourg) est validée, indexée, et documentée — prête à être exploitée par les notebooks de 
traitement Sentinel-2. La traçabilité complète vit dans `SOURCE.json`, `RECON.json`, `DB.json` 
et `INGESTION_REPORT.json`, tous versionnés à côté de ce notebook.